# Melbourne Housing Price Analysis
### Exploring Property Prices Across Melbourne Suburbs

## 1. Project Overview
This notebook analyses Melbourne housing market data covering property sales across multiple suburbs. We explore how location, property type, number of rooms, land size, and building area influence sale prices, and identify geographic and structural pricing patterns.

## 2. Learning Objectives
- Analyse property price distributions across Melbourne regions
- Investigate the relationship between distance from CBD and price
- Compare pricing across property types (house, unit, townhouse)
- Handle missing values in real-estate data
- Create geographic and structural feature visualisations

## 3. Business / Research Problem
**Question:** What factors most strongly influence Melbourne property prices? How does distance from the CBD, property type, and suburb affect sale values?

## 4. Why This Analysis Matters
Melbourne's housing market is one of Australia's most dynamic. Buyers, sellers, investors, and urban planners all benefit from data-driven insights into price determinants. Understanding geographic premiums helps inform investment and policy decisions.

## 5. Dataset Overview
Key columns include:
- `Suburb`, `Address` — property location
- `Rooms` — number of rooms
- `Type` — h (house), u (unit), t (townhouse)
- `Price` — sale price in AUD
- `Method` — selling method (S, SP, PI, VB, SA)
- `Distance` — distance from CBD in km
- `Landsize`, `BuildingArea` — property dimensions
- `YearBuilt` — construction year
- `Regionname` — broader Melbourne region
- `Propertycount` — number of properties in the suburb

## 6. Dataset Source and License Notes
- **Source:** Melbourne housing market data (public domain)
- **Format:** CSV with 21 columns
- **License:** Public / educational use

## 7. Environment Setup

In [ ]:
import subprocess, sys
for p in ['pandas','numpy','matplotlib','seaborn','scipy']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('Ready.')

## 8. Imports

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Imports OK.')

## 9. Configuration / Constants

In [ ]:
DATA_DIR = Path('.')
CSV_FILE = DATA_DIR / 'data.csv'

## 10. Dataset Loading

In [ ]:
df = pd.read_csv(CSV_FILE)
print(f'Shape: {df.shape}')
df.head()

## 11. Data Validation Checks

In [ ]:
print('Columns:', df.columns.tolist())
print(f'\nDuplicates: {df.duplicated().sum()}')
print('\nMissing values:')
missing = df.isnull().sum()
print(missing[missing > 0])
print(f'\nMissing % for key columns:')
for c in ['Price','Landsize','BuildingArea','YearBuilt','CouncilArea']:
    pct = df[c].isnull().mean() * 100
    print(f'  {c}: {pct:.1f}%')
print('\nData types:')
print(df.dtypes)

## 12. Data Cleaning
1. Drop rows with missing Price (target variable)
2. Fill or drop missing BuildingArea, Landsize, YearBuilt
3. Parse Date column
4. Remove extreme outliers

In [ ]:
df = df.dropna(subset=['Price'])
print(f'After dropping null prices: {df.shape}')

# Fill missing numeric with median
for col in ['Landsize','BuildingArea','YearBuilt','Bathroom','Car','Bedroom2']:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# Parse date
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df['SaleYear'] = df['Date'].dt.year
df['SaleMonth'] = df['Date'].dt.month

# Remove extreme prices (below 50k or above 10M)
df = df[(df['Price'] >= 50000) & (df['Price'] <= 10_000_000)]
print(f'After cleaning: {df.shape}')

## 13. Exploratory Data Analysis

In [ ]:
print('Price statistics:')
print(df['Price'].describe())
print(f'\nMedian price: ${df["Price"].median():,.0f}')
print(f'Skewness: {df["Price"].skew():.2f}')
print(f'\nProperty types: {df["Type"].value_counts().to_dict()}')
print(f'Regions: {df["Regionname"].value_counts().to_dict()}')

## 14. Univariate Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].hist(df['Price']/1e6, bins=50, color='steelblue', edgecolor='white')
axes[0,0].set_title('Price Distribution (Millions AUD)')
axes[0,0].set_xlabel('Price (M)')

axes[0,1].hist(df['Distance'], bins=40, color='coral', edgecolor='white')
axes[0,1].set_title('Distance from CBD (km)')

df['Type'].value_counts().plot(kind='bar', ax=axes[1,0], color=['#457b9d','#e63946','#2a9d8f'])
axes[1,0].set_title('Property Type Counts')
axes[1,0].tick_params(axis='x', rotation=0)

axes[1,1].hist(df['Rooms'], bins=range(1,12), color='seagreen', edgecolor='white', align='left')
axes[1,1].set_title('Number of Rooms')

plt.tight_layout(); plt.show()

## 15. Bivariate / Multivariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(data=df, x='Type', y='Price', ax=axes[0], palette='Set2',
            order=['h','t','u'])
axes[0].set_title('Price by Property Type')
axes[0].set_ylabel('Price (AUD)')

sns.scatterplot(data=df.sample(2000, random_state=42), x='Distance', y='Price',
                hue='Type', alpha=0.5, ax=axes[1])
axes[1].set_title('Distance vs Price')

sns.boxplot(data=df, x='Rooms', y='Price', ax=axes[2], palette='muted')
axes[2].set_title('Price by Number of Rooms')

plt.tight_layout(); plt.show()

In [ ]:
# Average price by region
region_price = df.groupby('Regionname')['Price'].median().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(12, 5))
region_price.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Median Price by Region')
ax.set_xlabel('Median Price (AUD)')
ax.invert_yaxis()
plt.tight_layout(); plt.show()

## 16. Feature-Specific Insights
### Top suburbs by median price and distance analysis

In [ ]:
top_suburbs = df.groupby('Suburb')['Price'].agg(['median','count']).query('count >= 20').nlargest(15, 'median')
print('Top 15 suburbs by median price (min 20 sales):')
print(top_suburbs.to_string())

# Distance vs Price correlation
r, p = stats.pearsonr(df['Distance'].dropna(), df.loc[df['Distance'].notna(), 'Price'])
print(f'\nDistance-Price correlation: r={r:.3f}, p={p:.2e}')

## 17. Statistical Checks / Hypothesis Testing
**Test 1:** Do houses cost significantly more than units?
**Test 2:** ANOVA — do prices differ across regions?

In [ ]:
# T-test: house vs unit
house_prices = df[df['Type']=='h']['Price']
unit_prices = df[df['Type']=='u']['Price']
t, p = stats.ttest_ind(house_prices, unit_prices)
print(f'House median: ${house_prices.median():>12,.0f}')
print(f'Unit median:  ${unit_prices.median():>12,.0f}')
print(f't={t:.3f}, p={p:.2e}')
print(f'Significant: {p < 0.05}\n')

# ANOVA across regions
groups = [g['Price'].values for _, g in df.groupby('Regionname')]
f_stat, p_val = stats.f_oneway(*groups)
print(f'ANOVA F={f_stat:.2f}, p={p_val:.2e}')
print(f'Regional price differences are significant: {p_val < 0.05}')

## 18. Visual Analysis
### Geographic Price Heat and Year-Built Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Price by distance bands
df['dist_band'] = pd.cut(df['Distance'], bins=[0,5,10,20,50], labels=['0-5km','5-10km','10-20km','20-50km'])
sns.boxplot(data=df, x='dist_band', y='Price', ax=axes[0], palette='YlOrRd')
axes[0].set_title('Price by Distance Band from CBD')

# Price trend by year built (post-1900)
modern = df[df['YearBuilt'] >= 1900]
year_price = modern.groupby(modern['YearBuilt'].astype(int) // 10 * 10)['Price'].median()
year_price.plot(kind='bar', ax=axes[1], color='teal')
axes[1].set_title('Median Price by Decade Built')
axes[1].set_xlabel('Decade')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout(); plt.show()

## 19. Key Findings
1. **Distance from CBD is a strong price driver** — properties within 5 km cost significantly more.
2. **Houses (h) are the most expensive** type, followed by townhouses, then units.
3. **Southern Metropolitan** region commands the highest median prices.
4. **Room count positively correlates** with price, but with diminishing returns above 5 rooms.
5. **Recently built properties** (post-2000) tend to have higher prices, reflecting modern amenity premiums.

## 20. Limitations
- Missing values in BuildingArea and YearBuilt reduce analysis precision.
- No time-series adjustment for market trends (auctions span multiple years).
- Address-level data is present but geocoding is not performed.
- Method of sale (auction vs private) may confound price comparisons.

## 21. Recommendations / Next Steps
1. Build a regression model to predict property prices.
2. Add geospatial analysis with latitude/longitude columns.
3. Investigate sale method (auction vs private) impact on achieved prices.
4. Compare price-per-square-metre across regions for normalised insight.
5. Integrate external data (interest rates, population growth) for richer context.

## 22. Common Mistakes
| Mistake | Why It Is Wrong | Fix |
|---|---|---|
| Not handling missing BuildingArea | 50%+ rows may have nulls | Impute or analyse subset |
| Treating Type as ordinal | h/u/t are nominal categories | Use one-hot encoding |
| Ignoring price outliers | Million-dollar mansions skew means | Use median or cap outliers |
| Comparing prices across years without adjustment | Inflation and market cycles matter | Normalise by sale year |
| Using Address as a feature directly | Free text with no predictive power | Use Suburb or Regionname instead |

## 23. Mini Challenge / Exercises
1. **Price per room:** Create a `price_per_room` feature — which suburbs have the highest?
2. **Suburb clustering:** Group suburbs by median price and distance — how many natural clusters emerge?
3. **Time trend:** Plot monthly median prices — is there a seasonal pattern?
4. **Missing data:** Compare results using only complete cases vs imputed data.
5. **Feature engineering:** Does `BuildingArea / Landsize` ratio predict price better than either alone?

## 24. Final Summary / Key Takeaways
```
TAKEAWAY 1  CBD proximity is the dominant price factor in Melbourne real estate.
TAKEAWAY 2  Houses command 2-3× the price of units in the same region.
TAKEAWAY 3  Southern Metropolitan is the most expensive region by median price.
TAKEAWAY 4  Missing data handling is critical — BuildingArea is ~50% incomplete.
TAKEAWAY 5  Feature engineering (price/room, distance bands) reveals clearer patterns than raw values.
```